In [4]:
from pathlib import Path
import sys

import pandas as pd


# ============================================================
# PROJECT CONFIG
# ============================================================

current = Path.cwd().resolve()

PROJECT_ROOT = next(
    p for p in [current, *current.parents]
    if (p / "src" / "data_pipeline" / "config.py").exists()
)

sys.path.insert(
    0,
    str(PROJECT_ROOT / "src" / "data_pipeline")
)

from config import (
    ANSWERS_DIR,
    INSIDERS_CSV,
)


# ============================================================
# LOAD CERT R4.2 GROUND TRUTH
# ============================================================

insiders = pd.read_csv(
    INSIDERS_CSV,
    dtype=str,
)

insiders["dataset"] = pd.to_numeric(
    insiders["dataset"],
    errors="coerce",
)

insiders["scenario"] = pd.to_numeric(
    insiders["scenario"],
    errors="coerce",
).astype("Int64")


r42 = insiders[
    insiders["dataset"] == 4.2
].copy()


# ============================================================
# PARSE INCIDENT DATES
# ============================================================

r42["start"] = pd.to_datetime(
    r42["start"],
    format="%m/%d/%Y %H:%M:%S",
    errors="coerce",
)

r42["end"] = pd.to_datetime(
    r42["end"],
    format="%m/%d/%Y %H:%M:%S",
    errors="coerce",
)

r42["duration_days"] = (
    r42["end"] - r42["start"]
).dt.total_seconds() / 86400


# ============================================================
# INCIDENT SUMMARY
# ============================================================

print("=" * 70)
print("CERT R4.2 INCIDENT SUMMARY")
print("=" * 70)

print(f"\nTotal incidents        : {len(r42)}")
print(
    f"Unique malicious users : "
    f"{r42['user'].nunique()}"
)


print("\nIncidents by scenario:")

scenario_counts = (
    r42["scenario"]
    .value_counts()
    .sort_index()
    .rename_axis("scenario")
    .reset_index(name="incident_count")
)

display(scenario_counts)


print("\nIncident duration statistics (days):")

duration_summary = (
    r42
    .groupby("scenario")["duration_days"]
    .agg(
        [
            "count",
            "min",
            "median",
            "mean",
            "max",
        ]
    )
    .round(2)
    .reset_index()
)

display(duration_summary)


print("\nGround-truth sample:")

display(
    r42[
        [
            "scenario",
            "user",
            "start",
            "end",
            "duration_days",
            "details",
        ]
    ].head()
)


print("\n✅ INCIDENT SUMMARY COMPLETE")

CERT R4.2 INCIDENT SUMMARY

Total incidents        : 70
Unique malicious users : 70

Incidents by scenario:


,scenario,incident_count
0,1,30
1,2,30
2,3,10



Incident duration statistics (days):


,scenario,count,min,median,mean,max
0,1,30,0.01,6.19,5.29,10.86
1,2,30,44.08,56.19,55.13,58.46
2,3,10,1.26,1.41,1.40,1.48



Ground-truth sample:


,scenario,user,start,end,duration_days,details
8,1,AAM0658,2010-10-23 01:34:19,2010-10-29 05:23:28,6.159132,r4.2-1-AAM0658.csv
9,1,AJR0932,2010-09-10 19:12:01,2010-09-18 02:02:51,7.285301,r4.2-1-AJR0932.csv
10,1,BDV0168,2010-07-30 19:56:44,2010-08-10 05:16:41,10.388854,r4.2-1-BDV0168.csv
11,1,BIH0745,2010-07-13 20:15:23,2010-07-13 21:20:44,0.045382,r4.2-1-BIH0745.csv
12,1,BLS0678,2010-09-21 01:16:22,2010-09-30 04:48:19,9.147187,r4.2-1-BLS0678.csv



✅ INCIDENT SUMMARY COMPLETE


In [6]:
# ============================================================
# INSPECT ANSWERS DIRECTORY
# ============================================================

print("=" * 70)
print("CERT ANSWERS DIRECTORY")
print("=" * 70)

print(f"\nAnswers directory:\n{ANSWERS_DIR}")


print("\nTop-level contents:")

for item in sorted(ANSWERS_DIR.iterdir()):
    print(f"  - {item.name}")


# ============================================================
# VERIFY EXAMPLE INCIDENT FILES
# ============================================================

example_users = [
    "AAM0658",   # Scenario 1 example
    "AAF0535",   # Known CERT case
]


for user in example_users:

    print(
        f"\nFiles associated with "
        f"{user}:"
    )

    matches = list(
        ANSWERS_DIR.rglob(
            f"*{user}*"
        )
    )

    if matches:
        for path in matches:
            print(
                f"  ✓ "
                f"{path.relative_to(ANSWERS_DIR)}"
            )

    else:
        print("  No matching files found.")


print("\n✅ ANSWERS DIRECTORY INSPECTION COMPLETE")

CERT ANSWERS DIRECTORY

Answers directory:
C:\Users\HP\Documents\FYP\Data_engineering\data\raw\cert_r4.2 dataset\answers

Top-level contents:
  - incident_event_audit.csv
  - insiders.csv
  - license.txt
  - r4.2-1
  - r4.2-2
  - r4.2-3
  - readme.txt
  - scenarios.txt

Files associated with AAM0658:
  ✓ r4.2-1\r4.2-1-AAM0658.csv

Files associated with AAF0535:
  ✓ r4.2-2\r4.2-2-AAF0535.csv

✅ ANSWERS DIRECTORY INSPECTION COMPLETE


In [8]:
# ============================================================
# INSPECT SAMPLE EXFILTRATION INCIDENTS
# ============================================================

print("=" * 70)
print("SAMPLE INCIDENT INSPECTION")
print("=" * 70)

# Scenario 1 and Scenario 2 are the exfiltration-relevant
# scenarios in CERT r4.2.
for scenario in [1, 2]:

    scenario_rows = r42[
        r42["scenario"] == scenario
    ]

    if scenario_rows.empty:
        print(
            f"\n❌ No incidents found "
            f"for Scenario {scenario}."
        )
        continue

    row = scenario_rows.iloc[0]

    filename = row["details"]

    # Expected CERT folder structure
    incident_file = (
        ANSWERS_DIR
        / f"r4.2-{scenario}"
        / filename
    )

    # Fallback search if folder structure differs
    if not incident_file.exists():

        matches = list(
            ANSWERS_DIR.rglob(filename)
        )

        incident_file = (
            matches[0]
            if matches
            else incident_file
        )


    print("\n" + "-" * 90)
    print(f"SCENARIO {scenario}")
    print("-" * 90)

    print(f"User         : {row['user']}")
    print(f"Attack start : {row['start']}")
    print(f"Attack end   : {row['end']}")
    print(
        f"Duration     : "
        f"{row['duration_days']:.2f} days"
    )
    print(f"Details file : {filename}")


    if not incident_file.exists():

        print(
            "\n❌ Incident details "
            "file not found."
        )

        continue


    print(
        f"File path    : "
        f"{incident_file.relative_to(ANSWERS_DIR)}"
    )

    print("\nFirst 30 raw event lines:\n")

    with incident_file.open(
        "r",
        encoding="utf-8",
        errors="replace",
    ) as file:

        for line_number, line in enumerate(
            file,
            start=1,
        ):

            if line_number > 30:
                break

            print(
                f"{line_number:02d} | "
                f"{line.rstrip()}"
            )


print("\n" + "=" * 70)
print("✅ INCIDENT INSPECTION COMPLETE")
print("=" * 70)

SAMPLE INCIDENT INSPECTION

------------------------------------------------------------------------------------------
SCENARIO 1
------------------------------------------------------------------------------------------
User         : AAM0658
Attack start : 2010-10-23 01:34:19
Attack end   : 2010-10-29 05:23:28
Duration     : 6.16 days
Details file : r4.2-1-AAM0658.csv
File path    : r4.2-1\r4.2-1-AAM0658.csv

First 30 raw event lines:

01 | logon,{K3V4-Y4OK65SI-1583GEOQ},10/23/2010 01:34:19,AAM0658,PC-9923,Logon
02 | device,{H1L0-X7RH83FI-5967VUQY},10/23/2010 06:18:48,AAM0658,PC-9923,Connect
03 | http,{Y4Q9-U5VQ11UG-1279ZPTL},10/23/2010 06:26:01,AAM0658,PC-9923,http://wikileaks.org/Julian_Assange/assange/The_Real_Story_About_DTAA/Gur_Erny_Fgbel_Nobhg_QGNN1528513805.php,spy bait bait distort evade surveillance deceit distort surveillance report subterfuge evade covert confidential covert top-secret clandestine isreal russia lie deceit china forgery covert europe covert aisa confidenti